# Training with genuinely unknown cell fate

**Role:** Training.

Train matched exclusive-fate FiLM models with a separate missing-fate indicator, including a zero-mask-rate control and several random masking rates. Unassigned is an observed cell identity, distinct from unknown fate. Reuse an explicitly selected size-conditioned training run's cohort and fitted transformations; save outer, inner fitting and early-stopping memberships, checkpoints, settings and histories. `benchmarks/masking_quality_and_robustness.ipynb` evaluates intact quality, single-neighbor masked quality and stability across seeds/rates; `ablation/fate_interventions.ipynb` compares masking with fate replacement and zeroing. No benchmarks or ablations are run during model fitting.

Timepoint selection is inherited from the reference run for both training and validation; there is no separate timepoint override. Choose a reference run trained on the desired cohort.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Model and missing-fate representation

Train exclusive-fate FiLM models at a configurable depth and width, with one channel per saved fate marker plus a binary **fate_missing** flag. Observed unassigned cells have zero fate channels and flag 0; hidden identities have zero fate channels and flag 1. Every input branch receives this encoding. Depth 0 is a cell-only control, with no message passing or FiLM layers.

Each run uses one depth/width configuration, evaluated across the selected folds, seeds and masking rates. Change the settings below for another architecture; the reference checkpoint supplies preprocessing and cohort membership, not pretrained weights. All masking models are trained from scratch, including the matched 0% control. Log cell count enters FiLM and the head; global geometry remains excluded from the head.

The separate benchmark notebook compares intact prediction quality, single-neighbor masked quality, and robustness across seeds/rates. These are conditional information effects, not causal cell interactions. Unknown identity can be inferred from context.


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Repository root used to resolve data and result paths.
if not (ROOT/'src').is_dir(): ROOT = ROOT.parent  # Repository root used to resolve data and result paths.
assert (ROOT/'src').is_dir()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.analysis.interventions.masking import MaskTrainingConfig, load_mask_benchmarks
# Study-specific functions are defined in this notebook.
from src.artifacts.paths import training_run_path, resolve_training_run
from src.artifacts.size_reference import size_reference
from src.artifacts.runs import AnalysisRun

## Training settings

Every optimizer step averages an intact and randomly masked pass with equal loss weight. In the masked pass, each node is hidden independently with the same probability, including naturally unassigned cells. Masks change every batch and do not use fate, N, curvature, geometry, or crypt labels. Rates share the same random uniforms, data order and base initialization within a seed. The 0% control also performs two passes per step, so dropout, normalization passes and optimizer-step counts are comparable. Additional mask-input weights start at zero.

Each original outer training fold is split into fitting and **inner early-stopping** organoids. All variants select checkpoints by intact inner-validation transformed-target MSE. Original outer validation organoids are reserved for the benchmarks and ablations. Cached baseline/target preprocessing is reused from the original outer-fold run; it is not refitted on the inner fitting subset. Existing cohort-level outlier interpolation is retained. This is not a claim of fully nested preprocessing or causal unbiasedness.

The original model used a different training/selection protocol. The newly trained **0% control** is the direct benchmark for masking augmentation; historical checkpoint comparisons remain secondary.

In [ ]:
# Execution
RUN_TRAINING = False  # True runs fitting when the training cell is executed; False stops before fitting.

SETTINGS = dict(
    # Run identity and reference data
    tag='',  # Optional purpose label for the timestamped masking run.
    reference_notebook='gin_depth_training',  # Workflow supplying an exclusive-fate FiLM reference.
    reference_run=None,  # Run name; None requires exactly one available run.
    reference_model_key=None,  # Select the source FiLM depth/width if its catalog contains several.

    # Saved outer splits and initialization
    folds=None,  # Saved fold IDs to include; None uses all. Cohort/timepoints/splits are inherited.
    seeds=None,  # Initialization seeds; None uses the reference seeds, or supply e.g. [42, 43, 44].

    # Model parameters (one architecture per run)
    depth=2,  # Number of message-passing layers; 0 gives a cell-only control.
    hidden_dim=128,  # Width of GNN hidden states and prediction head.
    film_hidden_dim=16,  # Width of the log-count FiLM conditioning network.
    dropout=.1,  # Hidden-feature dropout probability.
    norm='batch',  # Hidden normalization: batch, layer, or none.
    residual=True,  # Enable residual connections within the GNN.

    # Masking and inner early stopping
    rates=[0., .01, .02, .05],  # Include 0% control and at least one positive masking probability.
    masked_loss_weight=.5,  # Masked-pass loss fraction; intact fraction is one minus this value.
    inner_val_fraction=.15,  # Outer-training organoids reserved for early stopping.
    inner_split_seed=8123,  # Seed for the inner fitting/early-stopping split.

    # Training hyperparameters
    lr=3e-4,  # AdamW learning rate.
    weight_decay=1e-4,  # AdamW weight-decay coefficient.
    max_epochs=500,  # Maximum epochs per model.
    patience=30,  # Early-stopping patience on intact inner-validation MSE.
    batch_size=64,  # Graphs per optimization batch.
    num_workers=0,  # DataLoader worker processes.
    grad_clip=2.,  # Maximum gradient norm; None disables clipping.

    # Gaussian prediction loss and auxiliary edge penalty
    edge_loss_weight=.2,  # Edge-consistency penalty weight; 0 disables it.
    edge_loss_params=dict(
        weighted=False,  # Weight edges by target differences.
        alpha=2.,  # Exponent used for target-difference weighting.
        normalize_by='graph_std',  # Normalization for edge weights.
        clip_weight=4.,  # Maximum edge weight.
    ),
)

# Runtime and output locations
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Training device.
TORCH_THREADS = 4  # CPU threads used by PyTorch.
torch.set_num_threads(TORCH_THREADS)
REFERENCE_RUN = resolve_training_run(ROOT, SETTINGS['reference_notebook'], SETTINGS['reference_run'])
display(AnalysisRun(REFERENCE_RUN).records)
REFERENCE_RUN = size_reference(REFERENCE_RUN, SETTINGS['reference_model_key'], root=ROOT)
MASKING_RUN = training_run_path(ROOT, 'fate_masking_training', SETTINGS['tag'])  # New timestamped output directory.
TRAINING = MaskTrainingConfig(**{key: SETTINGS[key] for key in MaskTrainingConfig.__dataclass_fields__})
MODEL_OVERRIDES = {key: SETTINGS[key] for key in (
    'depth', 'hidden_dim', 'film_hidden_dim', 'dropout', 'norm', 'residual',
    'lr', 'weight_decay', 'edge_loss_weight', 'edge_loss_params')}
print('Device:', DEVICE)
print('Output:', MASKING_RUN)


## Train the models

This is the expensive training step. Completed checkpoints are reused; unfinished individual models restart their training. Settings/dependency fingerprints prevent silently combining different runs. Changing training settings requires a new output directory. The existing trained models and datasets are not overwritten.

The original Gaussian loss and edge penalty are retained. The model is trained to predict curvature with missing information; there is no fate-reconstruction loss or forced agreement between intact and masked predictions.
`RUN_TRAINING` is an execution guard, not a model option. False stops at the training cell; True fits and saves the selected models. It does not launch the separate analysis notebooks. Re-running the settings cell creates a new timestamped destination.


In [ ]:
from src.artifacts.provenance import workflow_fingerprint
import copy
from dataclasses import asdict, dataclass
import hashlib
import json
from pathlib import Path
import pickle
import shutil
import numpy as np
import pandas as pd
from scipy.stats import norm
import torch
from torch import nn
from torch_geometric.loader import DataLoader
from src.analysis.size_conditioning.cohort_inputs import make_model, seed_all, load_cohort, fold_graphs
from src.analysis.interventions.replacement import identity_codes, make_cases, _write_csv, UNASSIGNED
from src.analysis.interventions.size_sweeps import _size_override
from src.analysis.interventions.perturbation import predict_subgraph_center_distribution
from src.data.metadata import load_marker_names_from_dir
from src.data.subgraphs import build_ego_subgraphs_for_graph
from src.graph.neighborhood import compute_hop_rings
from src.inference.predict import predict_targets
from src.training.losses import CompositeLoss, make_base_loss, WeightedLossTerm, edge_loss_term
from src.analysis.interventions.masking import MaskTrainingConfig, rate_tag, encode_fates, random_mask, ObservedFateAdapter, _forward_with_mask, make_mask_model, inner_split, _loss, intact_validation_mse, train_mask_model, checkpoint_path, load_mask_checkpoint, _sha, checked_settings, score_arrays, benchmark_intact, evaluate_single_mask, make_quality_cases, load_mask_benchmarks
WORKFLOW_SOURCE = PROJECT_ROOT / 'experiments/training/fate_masking_training.ipynb'  # Notebook used when recording the code fingerprint.


### Train masking experiment

The study sequence is defined here; shared numerical operations are imported above.

In [ ]:
from src.artifacts import pickle_compat as artifact_pickle
def train_masking_experiment(root, reference_run, output_dir, *, config=MaskTrainingConfig(),
                             folds=None, seeds=None, device=None, run_tag='', model_overrides=None):
    root, reference, out = map(lambda p: Path(p).resolve(), (root, reference_run, output_dir))
    settings = json.loads((reference / 'settings.json').read_text())
    if settings.get('FEATURE_ENCODING') != 'exclusive_ordered':
        raise ValueError('Expected exclusive-fate reference data.')
    # Only architecture/optimization may change; preserve saved data and transforms.
    mapping = dict(depth='NUM_LAYERS', hidden_dim='HIDDEN_DIM', film_hidden_dim='FILM_HIDDEN_DIM',
                   dropout='DROPOUT', norm='NORM', residual='RESIDUAL', lr='LR',
                   weight_decay='WEIGHT_DECAY', edge_loss_weight='EDGE_LOSS_WEIGHT',
                   edge_loss_params='EDGE_LOSS_PARAMS')
    overrides = dict(model_overrides or {})
    if set(overrides) - set(mapping):
        raise ValueError(f'Unknown model settings: {sorted(set(overrides) - set(mapping))}')
    settings.update({mapping[key]: value for key, value in overrides.items()})
    for key, minimum in [('NUM_LAYERS', 0), ('HIDDEN_DIM', 1), ('FILM_HIDDEN_DIM', 1)]:
        if not isinstance(settings[key], int) or isinstance(settings[key], bool) or settings[key] < minimum:
            raise ValueError(f'{key} must be an integer >= {minimum}.')
    if (not 0 <= settings['DROPOUT'] < 1 or settings['NORM'] not in ('batch', 'layer', 'none')
            or not isinstance(settings['RESIDUAL'], bool) or settings['LR'] <= 0
            or settings['WEIGHT_DECAY'] < 0 or settings['EDGE_LOSS_WEIGHT'] < 0):
        raise ValueError('Invalid normalization, dropout, residual or optimizer/loss setting.')
    if settings['MODEL_GLOBAL_FEATURES']['gin_film_size'] != ['log_num_cells']:
        raise ValueError('Expected log(N) only, without geometry in the head.')
    splits = json.loads((reference / 'splits.json').read_text())
    folds = list(folds if folds is not None else [s['fold'] for s in splits])
    seeds = list(seeds if seeds is not None else settings['MODEL_SEEDS'])
    if (not folds or not seeds or len(set(folds)) != len(folds) or len(set(seeds)) != len(seeds)
            or not set(folds) <= {s['fold'] for s in splits}):
        raise ValueError('Invalid fold/seed selection.')
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = root / 'training_data' / settings['DATASET_NAME']
    markers = settings.get('MARKER_NAMES') or load_marker_names_from_dir(str(dataset))
    artifacts = ['settings.json', 'splits.json', 'tables/cohort.csv', 'geometric_normalization/references.csv']
    artifacts += [f'checkpoints/fold_{fold}_preprocessing.pkl' for fold in folds]
    if (reference/'cohort/manifest.json').exists():
        artifacts.append('cohort/manifest.json')
    dependencies = {name: _sha(reference / name) for name in artifacts}
    data_stats = [(p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in sorted(dataset.iterdir())] if not (reference/'cohort/manifest.json').exists() else []
    checked_settings(out / 'settings.json', dict(tag=run_tag, reference_run=str(reference), markers=markers, folds=folds,
        seeds=seeds, training=asdict(config), model_settings=settings, model_overrides=overrides, dependencies=dependencies,
        data_stat_sha256=hashlib.sha256(json.dumps(data_stats).encode()).hexdigest(), code_sha256=workflow_fingerprint(WORKFLOW_SOURCE)))
    for folder in ('checkpoints', 'history', 'benchmarks', 'figures'):
        (out / folder).mkdir(exist_ok=True)
    for name in artifacts:
        target = out / 'reference' / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if not (name.endswith('_preprocessing.pkl') and target.exists()):
            shutil.copy2(reference / name, target)
    if (reference / 'cohort' / 'manifest.json').exists():
        shutil.copytree(reference / 'cohort', out / 'reference' / 'cohort', dirs_exist_ok=True)
    from src.analysis.metrics.evaluation import compare_baseline_mse
    from src.training.preparation import prepare_fold, global_values
    baseline_rows, validation_rows = [], []
    graphs = load_cohort(dataset, settings, reference)
    for g in graphs:
        identity_codes(g.x)
    for split in splits:
        fold = split['fold']
        if fold not in folds:
            continue
        with (out / 'reference' / f'checkpoints/fold_{fold}_preprocessing.pkl').open('rb') as f:
            pre = artifact_pickle.load(f)
        # Preserve the reference target convention; baseline comparison is independent.
        pre.setdefault('residualized', bool(settings.get('SUBTRACT_CONSTANT_GLOBAL_BASELINE', pre.get('baseline') is not None)))
        if 'baseline_validation_mse' not in pre:
            if pre.get('baseline') is None:
                comparison = prepare_fold(graphs, split, global_features=[], residualize=False,
                    baseline_features=settings.get('BASELINE_FEATURES', ['log_num_cells', 'log_surface_area', 'log_volume', 'log_volume_over_area']),
                    baseline_kwargs=dict(max_epochs=config.max_epochs, patience=config.patience,
                        batch_size=config.batch_size, num_workers=config.num_workers, train_kwargs=dict(device=device)))
                pre['comparison_baseline'] = comparison['baseline']
                pre['baseline_predictions'] = comparison['baseline_predictions']
                pre['comparison_baseline_features'] = comparison['baseline_features']
                pre['comparison_global_center'] = comparison['global_center']
                pre['comparison_global_scale'] = comparison['global_scale']
                pre['baseline_validation_mse'] = comparison['baseline_validation_mse']
            else:
                # Historical references already contain a baseline but may lack its scores.
                val_baseline = [copy.deepcopy(graphs[i]) for i in split['val_indices']]
                names = pre.get('global_feature_names', ['log_num_cells', 'log_surface_area', 'log_volume', 'log_volume_over_area'])
                features = pre.get('baseline_features', settings.get('BASELINE_FEATURES', names))
                columns = [names.index(name) for name in features]
                for g in val_baseline:
                    values = (torch.tensor(global_values(g, names), dtype=torch.float32)
                              - torch.as_tensor(pre['global_center'])) / torch.as_tensor(pre['global_scale'])
                    g.global_feat = values[columns].reshape(1, -1)
                pre['baseline'].prediction_cache.clear()
                pre['baseline'].transform_graphs(val_baseline, in_place=True)
                pre['baseline_validation_mse'] = [dict(organoid_str=g.organoid_str, n_cells=len(g.x),
                    mse=float(g.y.square().mean())) for g in val_baseline]
                pre['baseline'].prediction_cache.clear()
            # Save new comparison metadata locally, without modifying the reference run.
            with (out/f'reference/checkpoints/fold_{fold}_preprocessing.pkl').open('wb') as f:
                pickle.dump(pre, f)
        baseline_rows.extend(dict(fold=fold, **row) for row in pre['baseline_validation_mse'])
        pd.DataFrame(baseline_rows).to_csv(out/'baseline_validation_mse.csv', index=False)
        groups = fold_graphs(graphs, split, pre, settings)

        def record_validation(model, rate, seed):
            intact = benchmark_intact(groups['val'], model, pre['residual_transform'], markers,
                                      batch_size=config.batch_size, device=device)
            for row in intact[intact.center_marker == 'All'].itertuples():
                validation_rows.append(dict(fold=fold, seed=seed, rate=rate, organoid_str=row.organoid_str,
                                            n_cells=row.n_nodes, mse=row.mse_mu))
            compare_baseline_mse(validation_rows, baseline_rows).to_csv(out/'validation_mse.csv', index=False)

        fit_ids, early_ids = inner_split(len(groups['train']), config.inner_val_fraction, config.inner_split_seed + fold)
        fit = [groups['train'][i] for i in fit_ids]; early = [groups['train'][i] for i in early_ids]
        membership = dict(fit=[g.organoid_str for g in fit], early_stopping=[g.organoid_str for g in early],
                          benchmark=[g.organoid_str for g in groups['val']])
        assert not (set(membership['fit']) & set(membership['early_stopping']))
        assert not ((set(membership['fit']) | set(membership['early_stopping'])) & set(membership['benchmark']))
        (out / f'fold_{fold}_membership.json').write_text(json.dumps(membership, indent=2))
        for rate in config.rates:
            for seed in seeds:
                path = checkpoint_path(out, fold, seed, rate)
                if path.exists():
                    print(f'Reuse completed checkpoint: {path.name}', flush=True)
                    model = load_mask_checkpoint(out, fold, seed, rate, settings, markers)
                    record_validation(model, rate, seed)
                    del model
                    continue
                print(f'Train fold={fold}, seed={seed}, masking rate={rate:g}', flush=True)
                model = make_mask_model(settings, markers, seed=seed)
                model, history = train_mask_model(model, fit, early, settings, config,
                                                  rate=rate, seed=seed, device=device)
                payload = dict(model_state={k:v.detach().cpu() for k,v in model.state_dict().items()},
                               history=history.to_dict('records'), rate=float(rate), fold=fold, seed=seed)
                temp = path.with_suffix('.tmp'); torch.save(payload, temp); temp.replace(path)
                history.to_csv(out / 'history' / f'{path.stem}.csv', index=False)
                record_validation(model, rate, seed)
                del model
    (out / 'training_complete.json').write_text(json.dumps(dict(folds=folds, seeds=seeds, rates=list(config.rates))))
    return out

In [ ]:
if not RUN_TRAINING:
    raise RuntimeError('Review settings, then set RUN_TRAINING=True to fit and save a new run.')
train_masking_experiment(ROOT, REFERENCE_RUN, MASKING_RUN,
    config=TRAINING, folds=SETTINGS['folds'], seeds=SETTINGS['seeds'], device=DEVICE,
    run_tag=SETTINGS['tag'], model_overrides=MODEL_OVERRIDES)


## Inspect optimization histories

Review the saved losses and stopping trajectories before proceeding to the independent benchmark notebook. Curves here are training diagnostics, not outer-fold performance estimates.

The intact validation MSE is compared with the same outer-fold baseline. Its fitted model is copied from the reference (or fitted if an older reference lacks one) and saved with the masking run, independent of whether the targets were residualized.

In [ ]:
history_paths = sorted((MASKING_RUN/'history').glob('*.csv'))
for path in history_paths[:4]:
    display(pd.read_csv(path).tail())
validation = pd.read_csv(MASKING_RUN/'validation_mse.csv')
display(validation.groupby('rate')[['mse','baseline_mse','mse_minus_baseline']].mean())